In [2]:
import pandas as pd

In [3]:
date_columns = [
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

train = pd.read_csv("data/train.csv", parse_dates=date_columns)
test = pd.read_csv("data/test.csv", parse_dates=date_columns)
events = pd.read_csv("data/events.csv.gz", compression="gzip", parse_dates=["event_ts"])

In [4]:
train.shape, train.dtypes

((11091, 5),
 cookie_id                       str
 cookie_created_at    datetime64[us]
 window_start_ts      datetime64[us]
 window_end_ts        datetime64[us]
 target                        int64
 dtype: object)

In [5]:
train.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [6]:
test.shape, test.dtypes

((4909, 4),
 cookie_id                       str
 cookie_created_at    datetime64[us]
 window_start_ts      datetime64[us]
 window_end_ts        datetime64[us]
 dtype: object)

In [7]:
test.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


In [8]:
events.shape, events.dtypes

((328905, 14),
 cookie_id                   str
 event_ts         datetime64[us]
 eid                       int64
 event_name                  str
 platform                    str
 user_agent                  str
 item_id                 float64
 item_category               str
 item_location               str
 seller_type                 str
 search_query                str
 search_page             float64
 pointer_x               float64
 pointer_y               float64
 dtype: object)

In [9]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [10]:
events.isna().sum()

cookie_id             0
event_ts              0
eid                   0
event_name            0
platform              0
user_agent            0
item_id          114597
item_category     32920
item_location     23793
seller_type      133853
search_query     228503
search_page      228503
pointer_x        220365
pointer_y        220365
dtype: int64

Ключевые поля ивентов (`cookie_id`, `event_ts`, `eid`, `event_name`, `platform`, `user_agent`) не содержат пропусков. 

In [11]:
events["event_name"].value_counts()

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

In [12]:
nullable_columns = [
    "item_id",
    "item_category",
    "item_location",
    "seller_type",
    "search_query",
    "search_page",
    "pointer_x",
    "pointer_y",
]

event_stat = (
    events.groupby("event_name")[nullable_columns]
    .agg(lambda x: x.notna().mean())
    .reset_index()
)

event_stat

,event_name,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,captcha_shown,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.272200,0.272200
1,contact_chat_open,1.0,0.941224,0.969306,0.915265,0.0,0.0,0.308245,0.308245
2,contact_message_sent,1.0,0.938656,0.971113,0.909120,0.0,0.0,0.322623,0.322623
3,contact_phone_show,1.0,0.941234,0.971456,0.906769,0.0,0.0,0.323259,0.323259
4,favorite_add,1.0,0.936637,0.972072,0.908342,0.0,0.0,0.349467,0.349467
5,item_view,1.0,0.940654,0.969756,0.909806,0.0,0.0,0.326568,0.326568
6,login,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.353439,0.353439
7,photo_swipe,1.0,0.941753,0.968261,0.912041,0.0,0.0,0.341923,0.341923
8,search_results_view,0.0,0.940340,0.969005,0.000000,1.0,1.0,0.330462,0.330462
9,seller_page_view,1.0,0.941562,0.968913,0.911107,0.0,0.0,0.336149,0.336149


Анализ заполненности показывает, что большинство пропусков не случайны. Например, `item_id` отсутствует только у `captcha_shown`, `login` и `search_results_view`. Значит удалять пустые строки нельзя. 

In [13]:
group_stat = (
    events.groupby("platform")[nullable_columns]
    .agg(lambda x: x.notna().mean())
    .reset_index()
)

group_stat

,platform,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ANDROID,0.663982,0.902689,0.932128,0.602939,0.297160,0.297160,0.000000,0.000000
1,Android,0.664836,0.902045,0.928669,0.605860,0.294315,0.294315,0.000000,0.000000
2,IOS,0.669024,0.914390,0.938537,0.608049,0.295854,0.295854,0.000000,0.000000
3,WEB,0.641105,0.894333,0.923380,0.583742,0.311150,0.311150,0.581139,0.581139
4,Web,0.643697,0.897164,0.925849,0.585679,0.311248,0.311248,0.588030,0.588030
5,android,0.662065,0.902488,0.931687,0.602552,0.297683,0.297683,0.000000,0.000000
6,desktop,0.642577,0.899693,0.926322,0.585840,0.312721,0.312721,0.586929,0.586929
7,iOS,0.661696,0.912246,0.937912,0.604498,0.302615,0.302615,0.000000,0.000000
8,ios,0.658411,0.904365,0.927661,0.600539,0.300392,0.300392,0.000000,0.000000
9,iphone,0.655862,0.903729,0.932976,0.593712,0.310505,0.310505,0.000000,0.000000


In [14]:
events["platform"].value_counts()

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

В поле `platform` находятся значения, различающиеся только регистром, они встречаются систематически и в сопоставимых количествах, значит их нельзя объединять, потому что это может быть полезный сигнал для классификации

In [15]:
platform_event_stat = (
    events.groupby(["platform", "event_name"])    
    .agg(
        event_count = ("cookie_id", "size"),
        pointer_x_share = ('pointer_x', lambda x: x.notna().mean()),
        pointer_y_share = ('pointer_y', lambda x: x.notna().mean())
    )
    .reset_index()
)

platform_event_stat.pivot(
    index="platform",
    columns="event_name",
    values="pointer_x_share"
)

event_name,captcha_shown,contact_chat_open,contact_message_sent,contact_phone_show,favorite_add,item_view,login,photo_swipe,search_results_view,seller_page_view
platform,,,,,,,,,,
ANDROID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Android,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
IOS,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
WEB,0.397037,0.633379,0.626632,0.642366,0.643136,0.561673,0.654776,0.638652,0.575617,0.591174
Web,0.423445,0.639077,0.642857,0.629111,0.671136,0.568364,0.657485,0.636785,0.578200,0.613008
android,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
desktop,0.437754,0.617801,0.681081,0.641152,0.655332,0.577906,0.655889,0.627912,0.568846,0.599185
iOS,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ios,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [16]:
train_cookies = set(train["cookie_id"])
test_cookies = set(test["cookie_id"])
event_cookies = set(events["cookie_id"])

print(f"Пересечение train и test: {len(train_cookies & test_cookies)}")
print(f"Количество куки из train без событий: {len(train_cookies - event_cookies)}" )
print(f"Количество куки из test без событий: {len(test_cookies - event_cookies)}") 
print(f"Количество куки из event, которых нет ни в test ни в train: {len(event_cookies - train_cookies - test_cookies)}") 
print(f"Число дубликатов cookie_id в train: {train["cookie_id"].duplicated().sum()}")
print(f"Число дубликатов cookie_id в test: {test["cookie_id"].duplicated().sum()}")

Пересечение train и test: 0
Количество куки из train без событий: 0
Количество куки из test без событий: 0
Количество куки из event, которых нет ни в test ни в train: 0
Число дубликатов cookie_id в train: 0
Число дубликатов cookie_id в test: 0


In [17]:
target_stats = pd.concat([
    train["target"].value_counts(), 
    train["target"].value_counts(normalize=True)
], axis=1).reset_index()

target_stats

,target,count,proportion
0,0,10192,0.918943
1,1,899,0.081057


In [18]:
train_durations = (train["window_end_ts"] - train["window_start_ts"]).dt.days.unique()
test_durations = (test["window_end_ts"] - test["window_start_ts"]).dt.days.unique()

print("Длительности окон")
print(f"train: {', '.join(map(str, train_durations))} день")
print(f"test: {', '.join(map(str, test_durations))} день")

Длительности окон
train: 1 день
test: 1 день


In [19]:
window_columns = [
    "cookie_id",
    "window_start_ts",
    "window_end_ts"
]

cookie_windows = pd.concat(
    [
    train[window_columns], 
    test[window_columns],
    ],
    ignore_index=True
)

cookie_windows

,cookie_id,window_start_ts,window_end_ts
0,ck_54a059eb7d3ea68b,2026-04-06,2026-04-07
1,ck_7e4de46eeab82974,2026-04-06,2026-04-07
2,ck_9320229ef6304522,2026-04-06,2026-04-07
3,ck_30ccd25bc1714ed9,2026-04-06,2026-04-07
4,ck_a77c5f05948cdeef,2026-04-06,2026-04-07
...,...,...,...
15995,ck_8b772caed8f1541c,2026-04-26,2026-04-27
15996,ck_5403c76fe68a08ba,2026-04-26,2026-04-27
15997,ck_3e6833574edf1736,2026-04-26,2026-04-27
15998,ck_e527fbb9450a4581,2026-04-26,2026-04-27


In [20]:
events_windows = pd.merge(events, cookie_windows, on="cookie_id", how="left", validate="many_to_one")

events_windows.head(2)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN,2026-04-26,2026-04-27
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN,2026-04-20,2026-04-21


In [21]:
window_mask = (
    (events_windows["window_start_ts"] <= events_windows["event_ts"]) 
    & (events_windows["event_ts"] < events_windows["window_end_ts"])
)

print(f"Событий вне окна наблюдения: {(~window_mask).sum()}")

Событий вне окна наблюдения: 40779


In [22]:
before_window = events_windows["event_ts"] < events_windows["window_start_ts"]
after_window = events_windows["event_ts"] >= events_windows["window_end_ts"]

print(f"Событий до начала окна {before_window.sum()}")
print(f"Событий после конца окна {after_window.sum()}")

Событий до начала окна 0
Событий после конца окна 40779


In [23]:
events_in_window = events_windows.loc[window_mask].copy()

В `events` есть события за пределами заданного окна наблюдения. При построении признаков будем использовать только события, удовлетворяющие условию `window_start_ts` ≤  `event_ts` < `window_end_ts`, чтобы не использовать информацию вне разрешенного окна

In [24]:
filtered_cookies = set(events_in_window["cookie_id"])
missing_after_filter = (train_cookies | test_cookies) - filtered_cookies

print(f"Количество куки без событий после фильтрации: {len(missing_after_filter)}")

Количество куки без событий после фильтрации: 0


In [25]:
train_created_in_window = (
    (train["cookie_created_at"] >= train["window_start_ts"])
    & (train["cookie_created_at"] < train["window_end_ts"])
)

test_created_in_window = (
    (test["cookie_created_at"] >= test["window_start_ts"])
    & (test["cookie_created_at"] < test["window_end_ts"])
)

print(f"Куки в train создано внутри окна: {train_created_in_window.sum()}")
print(f"Куки в test создано внутри окна: {test_created_in_window.sum()}")
print(f"Куки в train создано после окончания окна: {(train["cookie_created_at"] >= train["window_end_ts"]).sum()}")
print(f"Куки в test создано после окончания окна: {(test["cookie_created_at"] >= test["window_end_ts"]).sum()}")

Куки в train создано внутри окна: 0
Куки в test создано внутри окна: 0
Куки в train создано после окончания окна: 0
Куки в test создано после окончания окна: 0
